# CastForge - ComfyUI on a rented GPUOne notebook, **Colab or Kaggle**, that stands ComfyUI up and hands your laptop a URL to talk to.**Before you run anything, turn the GPU on.**- **Colab** - Runtime > Change runtime type > **T4 GPU** (free), or L4/A100 on a paid tier- **Kaggle** - Settings > Accelerator > **GPU T4 x2** or P100, *and* Internet > **On**Then run the cells in order. The last one prints a `https://....trycloudflare.com` address. Give it tothe laptop:```python generate/remote_gpu.py --url https://xxxx.trycloudflare.com pingpython generate/remote_gpu.py --url https://xxxx.trycloudflare.com fabric "worn indigo denim, visible twill" --kind denim --name worn_denim```The laptop refuses a swatch with a seam in it, so you find that out here rather than on a character.

## 1 - Where are we, and is the GPU actually on?

In [ ]:
import os, pathlib, subprocess, sysON_KAGGLE = os.path.isdir("/kaggle")ON_COLAB = os.path.isdir("/content") and not ON_KAGGLEWORK = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")KEEP = None                      # somewhere a dropped session cannot take with itprint("running on:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "something else")try:    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],                         capture_output=True, text=True).stdout.strip()except FileNotFoundError:    out = ""if not out:    raise SystemExit("No GPU. Turn the accelerator on first - see the note at the top.")print(out)

## 2 - Somewhere the work survives a dropped sessionFree sessions end: 90 minutes idle on Colab, twelve hours at the outside. Anything written only to theinstance dies with it. On Colab that means Drive; on Kaggle, `/kaggle/working`, which is kept when youcommit the notebook.

In [ ]:
if ON_COLAB:    from google.colab import drive    drive.mount("/content/drive")    KEEP = pathlib.Path("/content/drive/MyDrive/castforge")else:    KEEP = WORK / "castforge"(KEEP / "swatches").mkdir(parents=True, exist_ok=True)print("keeping finished work in", KEEP)

## 3 - ComfyUI, the Manager, and the one node that makes a texture tile

In [ ]:
COMFY = WORK / "ComfyUI"if not COMFY.exists():    !git clone -q https://github.com/comfyanonymous/ComfyUI {COMFY}%cd {COMFY}!pip install -q -r requirements.txtNODES = COMFY / "custom_nodes"def grab(url):    name = url.rstrip("/").split("/")[-1]    if not (NODES / name).exists():        !git clone -q {url} {NODES / name}    print("node pack:", name)grab("https://github.com/ltdrdata/ComfyUI-Manager")# Without this one, a generated square has a hard join and every repeat of it shows a seam.# remote_gpu.py refuses such a swatch, so do not skip it.grab("https://github.com/spinagon/ComfyUI-seamless-tiling")

## 4 - A checkpointSDXL base is the safe default: it fits a free T4, it knows what cloth looks like, and it needs nolicence acceptance. Swap in FLUX or a material LoRA later - the laptop does not care what made thesquare, only whether it tiles.

In [ ]:
MODELS = COMFY / "models" / "checkpoints"MODELS.mkdir(parents=True, exist_ok=True)CKPT = MODELS / "sd_xl_base_1.0.safetensors"URL = ("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/"       "resolve/main/sd_xl_base_1.0.safetensors")if not CKPT.exists():    !wget -q --show-progress -O {CKPT} {URL}print(CKPT, "%.1f GB" % (CKPT.stat().st_size / 2 ** 30))

## 5 - Start it, and open a door to it`cloudflared` needs no account and no signup, and the address it prints works from your laptop straightaway. It is a NEW address every time you run this cell - paste the new one into `--url`.

In [ ]:
import re, threading, time, urllib.requestif not pathlib.Path("/usr/local/bin/cloudflared").exists():    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64    !chmod +x /usr/local/bin/cloudflaredcomfy = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",                          "--output-directory", str(KEEP / "swatches")],                         cwd=str(COMFY), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)up = Falsefor _ in range(90):                        # answer first, then open the door    time.sleep(2)    try:        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3)        up = True        break    except Exception:        passif not up:    raise SystemExit("ComfyUI did not start - run cell 3 again and look for install errors")print("ComfyUI is up")tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188",                           "--no-autoupdate"],                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)for line in tunnel.stdout:    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)    if found:        print()        print("=" * 70)        print("  Point the laptop at this:")        print()        print("  python generate/remote_gpu.py --url %s ping" % found.group(0))        print("=" * 70)        print()        breakthreading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

## 6 - Keep the session alive while you workA free session with nothing happening in the browser is a session about to end. Run this and leave thetab open; stop it when you are finished.

In [ ]:
import datetime, time# No carriage return anywhere in here. A notebook keeps its source as JSON, and a real CR inside a# line splits that line when the file is read back, which breaks the whole cell.try:    while True:        print("alive", datetime.datetime.now().strftime("%H:%M:%S"),              "- stop this cell when you have finished", flush=True)        time.sleep(300)except KeyboardInterrupt:    print("stopped")

---### When it goes wrong| what you see | what it means ||---|---|| `seamless tiling NOT installed` from `ping` | cell 3 did not finish - run it again || `the best of 3 still has a seam` | the node is installed but not being used: check `CircularVAEDecode` is in what `ping` lists || the address stops answering | the session dropped. Re-run cell 5 and use the NEW address || `cannot reach a ComfyUI` | cell 5 is not running, or you pasted the old address |Finished swatches are kept in `castforge/swatches` on Drive (Colab) or `/kaggle/working/castforge/swatches`(Kaggle), so a dropped session never loses one that was already made.